# BrainGT Hyperparameter Search — Google Colab

Runs Optuna hyperparameter search for **BrainGT** across the first 5 folds.  
Optimises for **subject-level Pearson r** (window predictions averaged per subject before scoring).

### Runtime-break resilience
| What is saved | Where | When |
|---|---|---|
| Optuna study (all trials) | `Drive/…/optuna_study.db` (SQLite) | Continuously — auto-resumes if runtime breaks |
| Results JSON | `Drive/…/search_results_latest.json` | After **every** completed trial |
| Best model weights | `Drive/…/best_model_checkpoint.pt` | Every time a new best avg r is found |
| Best config JSON | `Drive/…/best_config.json` | Every time a new best avg r is found |

### Before running
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `folds_data/` to Google Drive
3. Upload `GNN-mri/` project folder to Drive **or** set `USE_GITHUB = True`
4. Edit the **CONFIG cell** below
5. Run all cells — if the runtime breaks, just **Run All** again; it resumes from where it stopped

In [ ]:
# ================================================================
# CONFIG — edit these before running
# ================================================================

# Folder in Drive containing graphs_outer*.pkl files
DRIVE_FOLD_DIR   = '/content/drive/MyDrive/MRI_data'

# Project code: set USE_GITHUB=False if you uploaded GNN-mri/ to Drive
USE_GITHUB        = False
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/GNN-mri'
GITHUB_REPO_URL   = 'https://github.com/Xin-999/Movie-HCP_Brain_Graph-master-test'

# Search settings
N_TRIALS     = 50     # total Optuna trials (resumes if interrupted)
N_EPOCHS     = 50     # epochs per trial
USE_ENHANCED = False  # True = BrainGTEnhanced, False = base BrainGT

# Drive folder where all outputs are saved
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/hyperparameter_search_results/braingt'


# Hardware tuning — adjust if you change Colab runtime tier
NUM_WORKERS  = 8      # DataLoader workers (Colab Pro has plenty of RAM)
PIN_MEMORY   = True   # pin memory for faster GPU transfer
USE_AMP      = True   # mixed precision (big speedup on A100/V100)

In [ ]:
!rm -rf /content/GNN-mri

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create Drive output dir immediately so checkpoints can be saved at any point
from pathlib import Path
drive_out = Path(DRIVE_OUTPUT_DIR)
drive_out.mkdir(parents=True, exist_ok=True)
print(f'Drive mounted. Output dir: {drive_out}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Output dir: /content/drive/MyDrive/hyperparameter_search_results/braingt


## Step 2 — Install dependencies
PyTorch is pre-installed on Colab. We install PyG (matched to CUDA) and Optuna.

In [ ]:
import subprocess, torch

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])

torch_ver = torch.__version__.split('+')[0]
cuda_ver  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_ver}  |  CUDA build: {cuda_ver}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse -f {pyg_url}')
run('pip install -q optuna tqdm scikit-learn scipy dill plotly pandas')
print('Dependencies ready.')

PyTorch 2.10.0  |  CUDA build: cu128
Dependencies ready.


In [ ]:
!rm -rf /content/GNN-mri

## Step 3 — Set up project code

In [ ]:
import os, shutil, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/GNN-mri')

if USE_GITHUB:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    print(f'Cloning {GITHUB_REPO_URL} ...')
    os.system(f'git clone {GITHUB_REPO_URL} {PROJECT_ROOT}')
else:
    src = Path(DRIVE_PROJECT_DIR)
    assert src.exists(), f'Project folder not found at {src}. Check DRIVE_PROJECT_DIR.'
    if not PROJECT_ROOT.exists():
        print(f'Copying project from Drive ...')
        shutil.copytree(str(src), str(PROJECT_ROOT))
    else:
        print(f'Project already at {PROJECT_ROOT}, skipping copy.')

assert (PROJECT_ROOT / 'models').exists(), 'models/ not found'
assert (PROJECT_ROOT / 'utils').exists(),  'utils/ not found'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root ready: {PROJECT_ROOT}')

Copying project from Drive ...
Project root ready: /content/GNN-mri


## Step 4 — Copy fold data to local disk
Local disk I/O is ~10x faster than Drive during training.

In [ ]:
from pathlib import Path
import shutil

LOCAL_FOLD_DIR  = Path('/content/folds_data')
DRIVE_FOLD_PATH = Path(DRIVE_FOLD_DIR)

assert DRIVE_FOLD_PATH.exists(), (
    f'folds_data not found at {DRIVE_FOLD_PATH}.\n'
    f'Upload the folds_data/ folder to that Drive path first.'
)

if not LOCAL_FOLD_DIR.exists():
    print('Copying fold data from Drive to local disk ...')
    shutil.copytree(str(DRIVE_FOLD_PATH), str(LOCAL_FOLD_DIR))
else:
    print('Fold data already on local disk.')

fold_files = sorted(LOCAL_FOLD_DIR.glob('graphs_outer*.pkl'))
print(f'Found {len(fold_files)} fold files.')

Fold data already on local disk.
Found 5 fold files.


## Step 5 — Verify GPU + imports

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, json, random
from datetime import datetime
from pathlib import Path
import optuna
import os

from models.brain_gt import BrainGT
from models_enhanced.brain_gt_enhanced import BrainGTEnhanced
from utils.data_utils import (
    load_graphs_with_normalization,
    create_dataloaders,
    compute_metrics,
    aggregate_window_predictions,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')

    # Matmul precision — allows TF32 for all matmuls (2-3x speedup on A100/H100)
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    print('TF32 + high matmul precision enabled')

    # cuDNN benchmark — finds fastest kernel for fixed input shape (268 ROIs)
    torch.backends.cudnn.benchmark = True
    print('cuDNN benchmark enabled')
else:
    print('WARNING: No GPU — training will be very slow.')

# Detect H100/A100 for BFloat16 (more stable than FP16, no GradScaler needed)
AMP_DTYPE = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16
print(f'Device: {device}')
print(f'AMP dtype: {AMP_DTYPE} ({"no GradScaler needed" if AMP_DTYPE == torch.bfloat16 else "using GradScaler"})')
print(f'CPU Cores: {os.cpu_count()}')

GPU : NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
TF32 + high matmul precision enabled
cuDNN benchmark enabled
Device: cuda
AMP dtype: torch.bfloat16 (no GradScaler needed)
CPU Cores: 12


## Step 6 — Select folds + check existing progress

In [ ]:
from pathlib import Path

# First 5 folds
fold_paths = sorted(LOCAL_FOLD_DIR.glob('graphs_outer*.pkl'))[:5]
model_type_label = 'enhanced' if USE_ENHANCED else 'base'
LOCAL_OUTPUT_DIR = Path('/content/braingt_search_results')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Folds selected ({len(fold_paths)}):')
for fp in fold_paths:
    print(f'  {fp.name}')

# ---- Preload all fold data into RAM (uses ~a few GB, saves disk I/O every trial) ----
print('\nPreloading fold data into RAM ...')
FOLD_CACHE = {}
for fp in fold_paths:
    train_g, val_g, test_g, info = load_graphs_with_normalization(
        str(fp), normalize_method='standard'
    )
    FOLD_CACHE[str(fp)] = (train_g, val_g, test_g, info)
    print(f'  {fp.name}: {len(train_g)} train, {len(val_g)} val, {len(test_g)} test windows')

import sys
cache_mb = sys.getsizeof(FOLD_CACHE) / 1e6
print(f'All folds cached in RAM. (avoids {N_TRIALS * len(fold_paths)} disk reads)')

# Check if there is an existing study in Drive (from a previous run)
db_path   = drive_out / 'optuna_study.db'
study_name = f'braingt_{model_type_label}_search'

if db_path.exists():
    _existing = optuna.load_study(
        study_name=study_name,
        storage=f'sqlite:///{db_path}',
    )
    n_done    = len([t for t in _existing.trials if t.state == optuna.trial.TrialState.COMPLETE])
    n_pruned  = len([t for t in _existing.trials if t.state == optuna.trial.TrialState.PRUNED])
    n_remain  = max(0, N_TRIALS - len(_existing.trials))
    print(f'\nExisting study found: {n_done} completed, {n_pruned} pruned, {n_remain} remaining.')
    if n_done > 0:
        bv = _existing.best_value
        print(f'Current best avg subj_r = {bv:.4f}')
    del _existing
else:
    print('\nNo existing study — will start fresh.')

Folds selected (5):
  graphs_outer1_inner1.pkl
  graphs_outer1_inner2.pkl
  graphs_outer1_inner3.pkl
  graphs_outer1_inner4.pkl
  graphs_outer1_inner5.pkl

Preloading fold data into RAM ...
Loading fold from /content/folds_data/graphs_outer1_inner1.pkl
Target normalization (standard):
  Original range: [66.49, 132.49]
  Normalized range: [-2.73, 2.35]
  Mean: 0.0000, Std: 1.0000
#train graphs (windows): 10440
#val   graphs (windows): 2700
#test  graphs (windows): 3330
  graphs_outer1_inner1.pkl: 10440 train, 2700 val, 3330 test windows
Loading fold from /content/folds_data/graphs_outer1_inner2.pkl
Target normalization (standard):
  Original range: [66.49, 132.49]
  Normalized range: [-2.55, 2.23]
  Mean: -0.0000, Std: 1.0000
#train graphs (windows): 10530
#val   graphs (windows): 2610
#test  graphs (windows): 3330
  graphs_outer1_inner2.pkl: 10530 train, 2610 val, 3330 test windows
Loading fold from /content/folds_data/graphs_outer1_inner3.pkl
Target normalization (standard):
  Origina

## Step 7 — Callback (auto-save after every trial)

The `SearchCallback` runs after every completed trial:  
- Saves `search_results_latest.json` to Drive  
- Saves `best_config.json` and `best_model_checkpoint.pt` when a new best r is found

In [ ]:
# Shared dict — objective writes best model state here;
# callback reads it and persists to Drive.
_search_state = {
    'best_r'         : -float('inf'),
    'best_state_dict': None,   # model weights (CPU tensors)
    'best_config'    : None,   # trial.params dict
    'best_trial_num' : None,
}


class SearchCallback:
    """Persists results and checkpoints to Drive after every Optuna trial."""

    def __init__(self, output_dir, fold_paths):
        self.output_dir = Path(output_dir)
        self.fold_paths = fold_paths

    def __call__(self, study, trial):
        # Always save full results JSON
        self._save_results(study)

        # Save checkpoint when this trial is the new study best
        if (
            trial.state == optuna.trial.TrialState.COMPLETE
            and trial.value is not None
            and trial.value >= study.best_value
        ):
            self._save_best_config(study)
            self._save_best_checkpoint(trial)
            print(f'  -> New best! avg_r={trial.value:.4f}  checkpoint saved to Drive.')

    # ------------------------------------------------------------------
    def _save_results(self, study):
        """Write all trial data to JSON on Drive."""
        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        best = study.best_trial if completed else None

        payload = {
            'model'      : 'braingt',
            'model_type' : model_type_label,
            'timestamp'  : datetime.now().isoformat(),
            'n_trials_target' : N_TRIALS,
            'n_trials_done'   : len(completed),
            'n_folds'    : len(self.fold_paths),
            'folds'      : [fp.name for fp in self.fold_paths],
            'best_avg_trial': {
                'trial_number' : best.number,
                'avg_r'        : best.user_attrs.get('avg_val_r', best.value),
                'std_r'        : best.user_attrs.get('std_val_r', 0),
                'fold_r_scores': best.user_attrs.get('fold_val_r_scores', []),
                'params'       : best.params,
            } if best else {},
            'all_trials' : [
                {
                    'number'       : t.number,
                    'value'        : t.value,
                    'state'        : str(t.state),
                    'params'       : t.params,
                    'avg_r'        : t.user_attrs.get('avg_val_r', t.value),
                    'std_r'        : t.user_attrs.get('std_val_r', 0),
                    'fold_r_scores': t.user_attrs.get('fold_val_r_scores', []),
                }
                for t in study.trials
            ],
        }
        path = self.output_dir / 'search_results_latest.json'
        with open(path, 'w') as f:
            json.dump(payload, f, indent=2)

    def _save_best_config(self, study):
        """Write best hyperparameters to a small easy-to-read JSON."""
        best = study.best_trial
        config = {
            'trial_number'  : best.number,
            'avg_val_r'     : best.user_attrs.get('avg_val_r', best.value),
            'std_val_r'     : best.user_attrs.get('std_val_r', 0),
            'fold_r_scores' : best.user_attrs.get('fold_val_r_scores', []),
            'params'        : best.params,
            'saved_at'      : datetime.now().isoformat(),
        }
        path = self.output_dir / 'best_config.json'
        with open(path, 'w') as f:
            json.dump(config, f, indent=2)

    def _save_best_checkpoint(self, trial):
        """Save model weights + config for the current best trial."""
        if _search_state['best_state_dict'] is None:
            return
        checkpoint = {
            'state_dict'    : _search_state['best_state_dict'],
            'config'        : _search_state['best_config'],
            'avg_val_r'     : trial.value,
            'trial_number'  : trial.number,
            'fold_r_scores' : trial.user_attrs.get('fold_val_r_scores', []),
            'n_folds'       : len(self.fold_paths),
            'saved_at'      : datetime.now().isoformat(),
        }
        path = self.output_dir / 'best_model_checkpoint.pt'
        torch.save(checkpoint, path)


print('SearchCallback defined.')

SearchCallback defined.


## Step 8 — Objective function

In [ ]:
import time

def _build_model(trial, in_dim):
    hidden_dim           = trial.suggest_categorical('hidden_dim', [64, 128, 256])
    n_heads              = trial.suggest_categorical('n_heads', [2, 4, 8])
    n_transformer_layers = trial.suggest_int('n_transformer_layers', 2, 4)
    n_gnn_layers         = trial.suggest_int('n_gnn_layers', 1, 3)
    dropout              = trial.suggest_float('dropout', 0.1, 0.4)
    pool_type            = trial.suggest_categorical('pool_type', ['attention', 'mean'])

    if USE_ENHANCED:
        return BrainGTEnhanced(
            in_dim=in_dim, hidden_dim=hidden_dim, n_rois=268,
            n_transformer_layers=n_transformer_layers,
            n_gnn_layers=n_gnn_layers, n_heads=n_heads, dropout=dropout,
        )
    return BrainGT(
        in_dim=in_dim, hidden_dim=hidden_dim, n_rois=268,
        n_transformer_layers=n_transformer_layers,
        n_gnn_layers=n_gnn_layers, n_heads=n_heads,
        dropout=dropout, pool_type=pool_type,
    )


def objective(trial):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    trial_start = time.time()

    seed = 42
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = True

    lr           = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [32, 64, 128])

    # BF16: no GradScaler needed (same dynamic range as FP32)
    # FP16: needs GradScaler to avoid NaN
    use_bf16 = (AMP_DTYPE == torch.bfloat16)
    scaler = None if use_bf16 else torch.amp.GradScaler('cuda', enabled=USE_AMP)

    fold_val_r_scores = []
    trial_best_r      = -float('inf')
    trial_best_state  = None

    for fold_idx, fold_path in enumerate(fold_paths):
        fold_start = time.time()

        train_graphs, val_graphs, test_graphs, _ = FOLD_CACHE[str(fold_path)]

        train_loader, val_loader, _ = create_dataloaders(
            train_graphs, val_graphs, test_graphs,
            batch_size=batch_size, num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        in_dim = train_graphs[0].x.size(-1)

        try:
            model = _build_model(trial, in_dim).to(device)
        except Exception as e:
            raise optuna.TrialPruned()

        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.MSELoss()

        n_warmup = max(1, N_EPOCHS // 5)
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    optimizer, start_factor=0.1, end_factor=1.0, total_iters=n_warmup),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max(1, N_EPOCHS - n_warmup), eta_min=lr * 0.01),
            ],
            milestones=[n_warmup],
        )

        best_val_r       = -float('inf')
        best_fold_state  = None
        patience_counter = 0
        patience         = max(5, N_EPOCHS // 3)

        for epoch in range(N_EPOCHS):
            try:
                # Train with AMP (BF16 or FP16)
                model.train()
                for batch in train_loader:
                    batch = batch.to(device, non_blocking=True)
                    optimizer.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=AMP_DTYPE):
                        out   = model(batch)
                        preds = out[0] if isinstance(out, tuple) else out
                        loss  = criterion(preds, batch.y.float())

                    if use_bf16:
                        # BF16: direct backward, no scaler
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    else:
                        # FP16: use GradScaler
                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()

                # Validate
                model.eval()
                all_preds, all_targets, all_subj_ids = [], [], []
                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(device, non_blocking=True)
                        with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=AMP_DTYPE):
                            out   = model(batch)
                            preds = out[0] if isinstance(out, tuple) else out
                        all_preds.append(preds.float().cpu())
                        all_targets.append(batch.y.cpu())
                        if hasattr(batch, 'subject_id'):
                            all_subj_ids.append(batch.subject_id.cpu())

            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    print(f'  OOM trial {trial.number} fold {fold_idx+1}')
                    torch.cuda.empty_cache()
                    raise optuna.TrialPruned()
                raise

            scheduler.step()

            preds_np   = torch.cat(all_preds).numpy()
            targets_np = torch.cat(all_targets).numpy()
            val_r      = compute_metrics(preds_np, targets_np)['r']

            if all_subj_ids:
                sids = torch.cat(all_subj_ids).numpy().flatten()
                if len(sids) == len(preds_np):
                    sp, _ = aggregate_window_predictions(preds_np, sids)
                    st, _ = aggregate_window_predictions(targets_np, sids)
                    val_r = compute_metrics(sp, st).get('r', val_r)

            if val_r > best_val_r + 1e-5:
                best_val_r  = val_r
                patience_counter = 0
                best_fold_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1

            if patience_counter >= patience:
                break

            trial.report(best_val_r, fold_idx * N_EPOCHS + epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        fold_val_r_scores.append(best_val_r)
        fold_elapsed = time.time() - fold_start
        print(f'  Fold {fold_idx+1}/{len(fold_paths)}: val_subj_r = {best_val_r:.4f}  [{fold_elapsed:.0f}s]')

        if best_val_r > trial_best_r:
            trial_best_r     = best_val_r
            trial_best_state = best_fold_state

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        del model, optimizer, scheduler, train_loader, val_loader

    avg_val_r = float(np.mean(fold_val_r_scores))
    std_val_r = float(np.std(fold_val_r_scores))

    trial.set_user_attr('avg_val_r',         avg_val_r)
    trial.set_user_attr('std_val_r',         std_val_r)
    trial.set_user_attr('fold_val_r_scores', fold_val_r_scores)
    trial.set_user_attr('n_folds',           len(fold_paths))

    if avg_val_r > _search_state['best_r']:
        _search_state['best_r']          = avg_val_r
        _search_state['best_state_dict'] = trial_best_state
        _search_state['best_config']     = trial.params.copy()
        _search_state['best_trial_num']  = trial.number

    trial_elapsed = time.time() - trial_start
    trial_mins, trial_secs = divmod(int(trial_elapsed), 60)
    trial.set_user_attr('elapsed_seconds', trial_elapsed)
    print(f'Trial {trial.number}: avg subj_r = {avg_val_r:.4f} +/- {std_val_r:.4f}  [{trial_mins}m {trial_secs}s]')
    return avg_val_r


print('Objective defined.')

Objective defined.


In [ ]:
# Check 1: BF16 or FP16?
print(f'AMP_DTYPE: {AMP_DTYPE}')  # Should be torch.bfloat16

# Check 2: Matmul precision
print(f'Matmul precision: {torch.get_float32_matmul_precision()}')  # Should be 'high'

# Check 3: TF32 enabled?
print(f'TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}')  # Should be True
print(f'TF32 cuDNN:  {torch.backends.cudnn.allow_tf32}')        # Should be True
print(f'cuDNN bench: {torch.backends.cudnn.benchmark}')          # Should be True

# Check 4: Does objective use BF16? (check if GradScaler is skipped)
import inspect
src = inspect.getsource(objective)
print(f'Uses AMP_DTYPE: {"AMP_DTYPE" in src}')       # Should be True
print(f'Uses GradScaler always: {"GradScaler(\'cuda\', enabled=USE_AMP)" in src}')  # Should be False
print(f'Uses use_bf16 check: {"use_bf16" in src}')    # Should be True


AMP_DTYPE: torch.bfloat16
Matmul precision: high
TF32 matmul: True
TF32 cuDNN:  True
cuDNN bench: True
Uses AMP_DTYPE: True
Uses GradScaler always: True
Uses use_bf16 check: True


## Step 9 — Run search

**If the runtime breaks:** just re-run all cells.  
The study is stored in `optuna_study.db` on Drive — Optuna loads it automatically and continues from the last completed trial.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SQLite storage on Drive — persists across runtime breaks
db_path  = drive_out / 'optuna_study.db'
storage  = f'sqlite:///{db_path}'

study = optuna.create_study(
    study_name    = study_name,
    direction     = 'maximize',
    storage       = storage,
    load_if_exists= True,   # <-- resume if interrupted
    # HyperbandPruner is the correct pairing for TPESampler.
    # Benchmarks show it consistently outperforms MedianPruner:
    # more aggressive early termination -> more configs explored per budget.
    # min_resource=1 epoch, max_resource=N_EPOCHS, reduction_factor=3 (standard).
    pruner        = optuna.pruners.HyperbandPruner(
                        min_resource=1,
                        max_resource=N_EPOCHS,
                        reduction_factor=3,
                    ),
    sampler       = optuna.samplers.TPESampler(seed=42),
)

n_done    = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_remain  = max(0, N_TRIALS - len(study.trials))

print(f'Study: {study_name}')
print(f'Already completed: {n_done}  |  Remaining: {n_remain} / {N_TRIALS}')
if n_done > 0:
    print(f'Current best avg subj_r = {study.best_value:.4f}')
print('=' * 60)

if n_remain == 0:
    print('All trials already complete. Skip to Step 10 for results.')
else:
    callback = SearchCallback(drive_out, fold_paths)

    study.optimize(
        objective,
        n_trials          = n_remain,
        n_jobs            = 1,
        show_progress_bar = True,
        callbacks         = [callback],
    )
    print('Search complete (or interrupted — re-run to continue).')

Study: braingt_base_search
Already completed: 4  |  Remaining: 19 / 50
Current best avg subj_r = 0.4671


  0%|          | 0/19 [00:00<?, ?it/s]

  Fold 1/5: val_subj_r = 0.4271  [3052s]
  Fold 2/5: val_subj_r = 0.5989  [7282s]
  Fold 3/5: val_subj_r = 0.5282  [3275s]
  Fold 4/5: val_subj_r = 0.4247  [3643s]
  Fold 5/5: val_subj_r = 0.2760  [3096s]
Trial 31: avg subj_r = 0.4510 +/- 0.1093   7s]
  Fold 1/5: val_subj_r = 0.3716  [2353s]
  Fold 2/5: val_subj_r = 0.4976  [4533s]
  Fold 3/5: val_subj_r = 0.5595  [2264s]
  Fold 4/5: val_subj_r = 0.4293  [2367s]
  Fold 5/5: val_subj_r = 0.4132  [1751s]
Trial 35: avg subj_r = 0.4542 +/- 0.0665   8s]
[W 2026-03-29 17:23:00,376] Trial 40 failed with parameters: {'lr': 0.0004919842502188657, 'weight_decay': 9.01752761391286e-05, 'batch_size': 32, 'hidden_dim': 256, 'n_heads': 4, 'n_transformer_layers': 4, 'n_gnn_layers': 3, 'dropout': 0.17655245699591018, 'pool_type': 'mean'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(tr

KeyboardInterrupt: 

## Step 10 — Results summary

In [ ]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    print('No completed trials yet.')
else:
    best = study.best_trial
    print('=' * 60)
    print(f'BEST TRIAL #{best.number}  (out of {len(completed)} completed)')
    print(f'  Avg subject-level Pearson r : {best.user_attrs.get("avg_val_r", best.value):.4f}')
    print(f'  Std across folds            : {best.user_attrs.get("std_val_r", 0):.4f}')
    scores = best.user_attrs.get('fold_val_r_scores', [])
    print(f'  Per-fold r scores           : {[round(x,4) for x in scores]}')
    print()
    print('Best hyperparameters:')
    for k, v in best.params.items():
        print(f'  {k}: {v}')
    print('=' * 60)

    # Best single-fold r across all trials
    best_sf = max(
        ((t.number, fi, fr, fold_paths[fi].name, t.params)
         for t in completed
         for fi, fr in enumerate(t.user_attrs.get('fold_val_r_scores', []))),
        key=lambda x: x[2],
        default=None,
    )
    if best_sf:
        print(f'Best single-fold r = {best_sf[2]:.4f}  '
              f'(trial #{best_sf[0]}, {best_sf[3]})')

In [ ]:
# Final save — also writes visualisation HTML to Drive
import pandas as pd

# Trigger callback manually for final save
if completed:
    SearchCallback(drive_out, fold_paths)._save_results(study)
    print(f'Final results saved to Drive: {drive_out}/search_results_latest.json')
    print(f'Best config saved to Drive  : {drive_out}/best_config.json')
    print(f'Best checkpoint at          : {drive_out}/best_model_checkpoint.pt')

try:
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.update_layout(title='BrainGT — Optimisation History')
    fig1.show()
    fig1.write_html(str(drive_out / 'optimization_history.html'))

    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.update_layout(title='BrainGT — Hyperparameter Importances')
    fig2.show()
    fig2.write_html(str(drive_out / 'param_importances.html'))
    print('Visualisations saved to Drive.')
except Exception as e:
    print(f'Visualisation skipped: {e}')

# Top-5 table
df = study.trials_dataframe()
df['state'] = df['state'].astype(str)
done_df = df[df['state'].str.contains('COMPLETE')].copy()
if not done_df.empty:
    top5 = done_df.sort_values('value', ascending=False).head(5)
    cols = ['number', 'value'] + [c for c in top5.columns
                                   if c.startswith('params_') and c in top5.columns][:5]
    print('\nTop 5 trials:')
    print(top5[cols].rename(columns={'value': 'avg_subj_r'}).to_string(index=False))